# Módulo 3 · Catálogo de datos con AWS Glue Crawler (boto3)



| # | Ejercicio | Equivalente en la CLI |
|---|---|---|
| 0 | Verificar que los datos de `processed/ventas/` están en S3 | `aws s3 ls --recursive` |
| 1 | Crear el rol de servicio `AWSGlueServiceRole-curso` | `aws iam create-role` + `attach-role-policy` + `put-role-policy` |
| 2 | Crear la base de datos `ventas_db` en el Data Catalog | `aws glue create-database` |
| 3 | Crear el crawler `crawler-ventas` | `aws glue create-crawler` |
| 4 | Ejecutarlo y monitorear su estado | `aws glue start-crawler` + `get-crawler` |
| 5 | Revisar el esquema de la tabla `ventas` | `aws glue get-tables` / `get-table` |
| 6 | Revisar las particiones | `aws glue get-partitions` |
| 7 | Limpieza (opcional) | `delete-crawler`, `delete-database`, `delete-role` |

**Requisito:** haber ejecutado `01_s3_buckets_boto3.ipynb`, que sube las ventas en Parquet a `s3://<bucket>/processed/ventas/year=/month=/day=/`.

**Costo:** el Data Catalog es gratis hasta 1 M de objetos al mes. El crawler cobra **USD 0.44 por DPU-hora, con mínimo de 10 minutos** por ejecución: unos pocos centavos cada vez que lo ejecutes.

## Repaso: ¿qué vamos a construir?

```
 S3 (datos)                               Glue Data Catalog (metadatos)
 processed/ventas/                        Database: ventas_db
   year=2026/month=10/day=20/  ──crawler──►  Tabla: ventas
   ...                                        columnas + tipos
   year=2026/month=10/day=27/                 ubicación en S3, formato Parquet
                                              particiones year / month / day
```

- El **catálogo no copia datos**: solo describe dónde están y cómo leerlos. Borrar la tabla no borra los archivos.
- El **crawler** se conecta a S3 con un **rol IAM**, detecta el formato, deduce columnas y particiones, y crea o actualiza la tabla.

## Configuración

Usa el mismo `BUCKET` y la misma `REGION` del notebook 01. Tu cuenta solo permite **us-east-2**: una SCP de la organización bloquea us-east-1.

In [2]:
from pathlib import Path
import csv
import json
import time

import boto3
import pandas as pd
from botocore.exceptions import ClientError, ProfileNotFound

BUCKET = "de-tecsup-marcell"            # <- el mismo del notebook 01
REGION = "us-east-2"
PERFIL = "curso"

ROL_GLUE = "AWSGlueServiceRole-curso"   # el prefijo AWSGlueServiceRole- importa (ver ejercicio 1)
DATABASE = "ventas_db"
CRAWLER = "crawler-ventas"
RUTA_VENTAS = f"s3://{BUCKET}/processed/ventas/"

PROYECTO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
CREDENCIALES = PROYECTO / "credentials"

In [3]:
def leer_csv_credenciales(carpeta: Path) -> dict:
    """Lee el .csv que descarga IAM (columnas 'Access key ID' y 'Secret access key')."""
    archivo = next(carpeta.glob("*accessKeys*.csv"))
    with archivo.open(encoding="utf-8-sig") as f:
        fila = next(csv.DictReader(f))
    return {
        "aws_access_key_id": fila["Access key ID"],
        "aws_secret_access_key": fila["Secret access key"],
    }


try:
    session = boto3.Session(profile_name=PERFIL, region_name=REGION)
    origen = f"perfil '{PERFIL}' de ~/.aws/"
except ProfileNotFound:
    session = boto3.Session(**leer_csv_credenciales(CREDENCIALES), region_name=REGION)
    origen = "archivo .csv de credentials/"

identidad = session.client("sts").get_caller_identity()
CUENTA = identidad["Account"]
print("Credenciales tomadas de:", origen)
print("ARN:", identidad["Arn"])

s3 = session.client("s3")
iam = session.client("iam")      # IAM es global: no depende de la región
glue = session.client("glue")

Credenciales tomadas de: archivo .csv de credentials/
ARN: arn:aws:iam::073813807844:user/marcell-de


## Ejercicio 0 · ¿Están los datos en S3?

El crawler solo registra lo que encuentra. Deberías ver **8 archivos Parquet**, uno por día.

In [4]:
respuesta = s3.list_objects_v2(Bucket=BUCKET, Prefix="processed/ventas/")
keys = [o["Key"] for o in respuesta.get("Contents", [])]
for k in keys:
    print(k)

assert keys, "No hay datos en processed/ventas/: ejecuta primero el notebook 01 (ejercicio 6)"
print(f"\n{len(keys)} archivo(s) listos para el crawler")

processed/ventas/year=2026/month=10/day=20/ventas.parquet
processed/ventas/year=2026/month=10/day=21/ventas.parquet
processed/ventas/year=2026/month=10/day=22/ventas.parquet
processed/ventas/year=2026/month=10/day=23/ventas.parquet
processed/ventas/year=2026/month=10/day=24/ventas.parquet
processed/ventas/year=2026/month=10/day=25/ventas.parquet
processed/ventas/year=2026/month=10/day=26/ventas.parquet
processed/ventas/year=2026/month=10/day=27/ventas.parquet

8 archivo(s) listos para el crawler


## Ejercicio 1 · Rol de servicio para Glue

Glue **no usa tus access keys**: asume un **rol**. Un rol tiene dos partes:

| Parte | Pregunta que responde | En nuestro rol |
|---|---|---|
| **Trust policy** (política de confianza) | ¿**Quién** puede asumir el rol? | El servicio `glue.amazonaws.com` |
| **Permissions policies** (políticas de permisos) | ¿**Qué** puede hacer quien lo asume? | `AWSGlueServiceRole` (Glue) + una política *inline* para leer y escribir en nuestro bucket |

¿Por qué la política inline? La política administrada `AWSGlueServiceRole` solo da acceso a buckets que empiezan con `aws-glue-*`. Nuestro bucket se llama distinto, así que le damos acceso explícito, solo a ese bucket (mínimo privilegio).

¿Por qué el nombre empieza con `AWSGlueServiceRole-`? Para entregarle un rol a Glue necesitas el permiso `iam:PassRole`. La política `AWSGlueConsoleFullAccess` de tu usuario solo lo concede para roles con ese prefijo.

In [5]:
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "glue.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}

politica_bucket = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject", "s3:ListBucket"],
        "Resource": [
            f"arn:aws:s3:::{BUCKET}",      # ListBucket aplica al bucket
            f"arn:aws:s3:::{BUCKET}/*",    # Get/Put/Delete aplican a los objetos
        ],
    }],
}

print(json.dumps(politica_bucket, indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:PutObject",
        "s3:DeleteObject",
        "s3:ListBucket"
      ],
      "Resource": [
        "arn:aws:s3:::de-tecsup-marcell",
        "arn:aws:s3:::de-tecsup-marcell/*"
      ]
    }
  ]
}


In [6]:
# 1. Crear el rol con su trust policy
try:
    rol = iam.create_role(
        RoleName=ROL_GLUE,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Rol de servicio para los crawlers y jobs de Glue del curso",
    )["Role"]
    print("Rol creado:", rol["Arn"])
except iam.exceptions.EntityAlreadyExistsException:
    rol = iam.get_role(RoleName=ROL_GLUE)["Role"]
    print("El rol ya existía:", rol["Arn"])

# 2. Adjuntar la política administrada de Glue
iam.attach_role_policy(
    RoleName=ROL_GLUE,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole",
)

# 3. Política inline con acceso a nuestro bucket
iam.put_role_policy(
    RoleName=ROL_GLUE,
    PolicyName="acceso-bucket-curso",
    PolicyDocument=json.dumps(politica_bucket),
)

ROL_ARN = rol["Arn"]

El rol ya existía: arn:aws:iam::073813807844:role/AWSGlueServiceRole-curso


Verifiquemos cómo quedó el rol. Las políticas *administradas* y las *inline* se consultan por separado.

> IAM es **eventualmente consistente**: un rol recién creado puede tardar unos segundos en estar disponible para Glue. Por eso esperamos un momento antes de crear el crawler.

In [7]:
administradas = iam.list_attached_role_policies(RoleName=ROL_GLUE)["AttachedPolicies"]
inline = iam.list_role_policies(RoleName=ROL_GLUE)["PolicyNames"]

print("Rol:", ROL_ARN)
print("Confía en:", [s["Principal"] for s in rol["AssumeRolePolicyDocument"]["Statement"]])
print("Políticas administradas:", [p["PolicyName"] for p in administradas])
print("Políticas inline:", inline)

time.sleep(10)   # margen para que el rol se propague

Rol: arn:aws:iam::073813807844:role/AWSGlueServiceRole-curso
Confía en: [{'Service': 'glue.amazonaws.com'}]
Políticas administradas: ['AWSGlueServiceRole']
Políticas inline: ['acceso-bucket-curso']


## Ejercicio 2 · Base de datos del catálogo

Equivale a `aws glue create-database --database-input '{"Name": "ventas_db"}'`.

Una *database* del Data Catalog es solo un **espacio de nombres** que agrupa tablas. No es un servidor ni guarda datos.

In [8]:
try:
    glue.create_database(DatabaseInput={
        "Name": DATABASE,
        "Description": "Ventas del curso de Data Engineering (zona processed)",
    })
    print("Base de datos creada:", DATABASE)
except glue.exceptions.AlreadyExistsException:
    print("La base de datos ya existía:", DATABASE)

print([db["Name"] for db in glue.get_databases()["DatabaseList"]])

La base de datos ya existía: ventas_db
['default', 'ventas_db']


## Ejercicio 3 · Crear el crawler

Equivale a `aws glue create-crawler`.

| Parámetro | Valor | Para qué |
|---|---|---|
| `Role` | `AWSGlueServiceRole-curso` | Identidad con la que el crawler lee S3 |
| `DatabaseName` | `ventas_db` | Dónde registra las tablas |
| `Targets.S3Targets` | `s3://<bucket>/processed/ventas/` | Qué prefijo recorre. **La tabla toma el nombre del último prefijo: `ventas`** |
| `SchemaChangePolicy` | `UPDATE_IN_DATABASE` / `LOG` | Si cambia el esquema, actualiza la tabla; si desaparece un archivo, solo lo registra en el log |
| `RecrawlPolicy` | `CRAWL_EVERYTHING` | Recorre todo cada vez. La alternativa `CRAWL_NEW_FOLDERS_ONLY` solo agrega carpetas nuevas |

Sin `Schedule` el crawler se ejecuta **bajo demanda**. Con `Schedule="cron(0 2 * * ? *)"` correría todos los días a las 02:00 UTC.

In [9]:
config_crawler = dict(
    Name=CRAWLER,
    Role=ROL_GLUE,
    DatabaseName=DATABASE,
    Description="Registra la tabla ventas particionada por year/month/day",
    Targets={"S3Targets": [{"Path": RUTA_VENTAS}]},
    SchemaChangePolicy={"UpdateBehavior": "UPDATE_IN_DATABASE", "DeleteBehavior": "LOG"},
    RecrawlPolicy={"RecrawlBehavior": "CRAWL_EVERYTHING"},
)

for intento in range(1, 6):
    try:
        glue.create_crawler(**config_crawler)
        print("Crawler creado:", CRAWLER)
        break
    except glue.exceptions.AlreadyExistsException:
        glue.update_crawler(**config_crawler)
        print("El crawler ya existía: configuración actualizada")
        break
    except glue.exceptions.InvalidInputException as e:
        # Típico si el rol aún no se propagó: "Service is unable to assume provided role"
        print(f"Intento {intento}: {e.response['Error']['Message']}")
        time.sleep(10)
else:
    raise RuntimeError("No se pudo crear el crawler: revisa el rol del ejercicio 1")

El crawler ya existía: configuración actualizada


## Ejercicio 4 · Ejecutar y monitorear el crawler

El estado pasa por `RUNNING → STOPPING → READY`. Aunque los datos sean pequeños, **tarda 1–2 minutos**: Glue tiene que aprovisionar recursos antes de empezar.

In [10]:
try:
    glue.start_crawler(Name=CRAWLER)
    print("Crawler iniciado")
except glue.exceptions.CrawlerRunningException:
    print("El crawler ya estaba en ejecución")

inicio = time.time()
estado_anterior = None
while True:
    estado = glue.get_crawler(Name=CRAWLER)["Crawler"]["State"]
    if estado != estado_anterior:
        print(f"[{time.time() - inicio:5.0f} s] {estado}")
        estado_anterior = estado
    if estado == "READY" and time.time() - inicio > 5:
        break
    time.sleep(10)

Crawler iniciado
[    0 s] RUNNING
[   41 s] READY


`State = READY` solo dice que el crawler terminó, no que lo hizo bien. El resultado real está en `LastCrawl`, y las métricas dicen cuántas tablas creó o actualizó.

In [11]:
ultimo = glue.get_crawler(Name=CRAWLER)["Crawler"]["LastCrawl"]
print("Estado:", ultimo["Status"])            # SUCCEEDED | FAILED | CANCELLED
if ultimo["Status"] != "SUCCEEDED":
    print("Error:", ultimo.get("ErrorMessage"))
    print("Logs en CloudWatch:", ultimo.get("LogGroup"), "/", ultimo.get("LogStream"))

metricas = glue.get_crawler_metrics(CrawlerNameList=[CRAWLER])["CrawlerMetricsList"][0]
print("Tablas creadas:     ", metricas["TablesCreated"])
print("Tablas actualizadas:", metricas["TablesUpdated"])
print("Duración (s):       ", round(metricas["LastRuntimeSeconds"]))

Estado: SUCCEEDED
Tablas creadas:      0
Tablas actualizadas: 0
Duración (s):        38


## Ejercicio 5 · Revisar el esquema registrado

Equivale a `aws glue get-tables --database-name ventas_db --query 'TableList[].Name'`.

**Resultado esperado** (diapositiva 25): 1 tabla `ventas`, con las columnas del archivo más `year`, `month` y `day` como columnas de **partición**.

In [12]:
tablas = glue.get_tables(DatabaseName=DATABASE)["TableList"]
print("Tablas en", DATABASE, "->", [t["Name"] for t in tablas])

Tablas en ventas_db -> ['ventas', 'ventas_diarias']


In [13]:
tabla = glue.get_table(DatabaseName=DATABASE, Name="ventas")["Table"]
sd = tabla["StorageDescriptor"]

print("Ubicación:   ", sd["Location"])
print("Clasificación:", tabla["Parameters"].get("classification"))
print("InputFormat: ", sd["InputFormat"].split(".")[-1])
print("Filas aprox.: ", tabla["Parameters"].get("recordCount"))

esquema = pd.DataFrame(
    [{"columna": c["Name"], "tipo": c["Type"], "origen": "archivo"} for c in sd["Columns"]]
    + [{"columna": c["Name"], "tipo": c["Type"], "origen": "partición"} for c in tabla["PartitionKeys"]]
)
esquema

Ubicación:    s3://de-tecsup-marcell/processed/ventas/
Clasificación: parquet
InputFormat:  MapredParquetInputFormat
Filas aprox.:  678


,columna,tipo,origen
0,producto_id,bigint,archivo
1,nombre,string,archivo
2,categoria,string,archivo
3,precio_unitario,double,archivo
4,cantidad,bigint,archivo
5,pedido_id,string,archivo
6,fecha,string,archivo
7,cliente_id,bigint,archivo
8,canal,string,archivo
9,monto,double,archivo


Fíjate en los tipos:

- Las columnas del archivo traen el tipo que **pandas escribió en el Parquet**: `bigint` para enteros, `double` para decimales, `string` para texto.
- `fecha` queda como `string` porque en el CSV era texto. Se puede convertir en Athena con `CAST(fecha AS date)`.
- Las particiones `year`, `month` y `day` se infieren como **`string`**. Por eso en Athena se filtran con comillas: `WHERE year = '2026' AND day = '22'`.

## Ejercicio 6 · Revisar las particiones

Equivale a `aws glue get-partitions --database-name ventas_db --table-name ventas --query 'Partitions[].Values'`.

Cada partición apunta a una carpeta de S3. Esta lista es la que usa Athena para el **partition pruning**: con `WHERE day = '22'` solo lee esa carpeta.

In [14]:
particiones = []
for pagina in glue.get_paginator("get_partitions").paginate(DatabaseName=DATABASE, TableName="ventas"):
    particiones.extend(pagina["Partitions"])

particiones.sort(key=lambda p: p["Values"])
for p in particiones:
    print(p["Values"], "->", p["StorageDescriptor"]["Location"])

print(f"\n{len(particiones)} particiones (esperado: 8)")

['2026', '10', '20'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=20/
['2026', '10', '21'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=21/
['2026', '10', '22'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=22/
['2026', '10', '23'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=23/
['2026', '10', '24'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=24/
['2026', '10', '25'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=25/
['2026', '10', '26'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=26/
['2026', '10', '27'] -> s3://de-tecsup-marcell/processed/ventas/year=2026/month=10/day=27/

8 particiones (esperado: 8)


### ¿Y si llegan días nuevos?

El catálogo **no se entera solo** de las carpetas nuevas en S3. Hay tres formas de registrarlas:

| Opción | Cómo | Cuándo conviene |
|---|---|---|
| Volver a ejecutar el crawler | `glue.start_crawler(Name=CRAWLER)` | Sencillo, pero cobra mínimo 10 min de DPU por ejecución |
| `MSCK REPAIR TABLE ventas` | Consulta en Athena (módulo 4) | Athena recorre S3 buscando carpetas nuevas |
| `ALTER TABLE ... ADD PARTITION` o `glue.batch_create_partition` | Registrar exactamente la partición del día | Lo más barato y preciso: lo haremos desde Airflow (módulo 6) |

## Ejercicio 7 · Limpieza (opcional)


Borrar la base de datos elimina las tablas del **catálogo**, pero **no** los archivos de S3. El bucket se borra en el notebook 01.

In [ ]:
BORRAR = False

if BORRAR:
    glue.delete_crawler(Name=CRAWLER)
    glue.delete_database(Name=DATABASE)          # borra también sus tablas y particiones del catálogo

    # Un rol solo se puede borrar sin políticas adjuntas
    for p in iam.list_attached_role_policies(RoleName=ROL_GLUE)["AttachedPolicies"]:
        iam.detach_role_policy(RoleName=ROL_GLUE, PolicyArn=p["PolicyArn"])
    for nombre in iam.list_role_policies(RoleName=ROL_GLUE)["PolicyNames"]:
        iam.delete_role_policy(RoleName=ROL_GLUE, PolicyName=nombre)
    iam.delete_role(RoleName=ROL_GLUE)

    print("Crawler, base de datos y rol eliminados")